# Day 13(M3 Day04) 실습 3 — 하이브리드 검색과 재순위

**목표**: BM25(키워드)와 의미 검색을 하이브리드로 결합하고, 리랭킹과 문맥 압축으로 근거의 순서와 양을 다듬은 뒤, 검색 정확도로 기본 RAG와 개선 RAG를 비교한다.

| Part | 내용 |
| --- | --- |
| Part 1 | 하이브리드 검색 (BM25 + 의미 검색, RRF) — 토크나이저·가중치 오류 |
| Part 2 | LLM 리랭킹 — 사내 규정 조항 번호 도메인 반복 |
| Part 3 | 재순위·문맥 압축 — AI 트렌드 보고서로 Flashrank, 추출·필터, 파이프라인 |
| Part 4 | 검색 정확도 평가 — 기본 vs 개선, 면접 코치 지식 검색 평가 |

| Day13 실습 | 주제 |
| --- | --- |
| 실습 1 | Naive RAG의 한계와 문서 전처리 |
| 실습 2 | 질의 개선 |
| **실습 3 (이 노트북)** | 하이브리드 검색과 재순위 |
| 실습 4 | RAG 생성 품질 평가 |

> **참고:** 실습 전 가상환경 활성화, `.env`의 `OPENAI_API_KEY`·`DATABASE_URL`을 확인한다. `uv add rank-bm25 flashrank`가 필요하다(이미 설치돼 있다면 생략). Part 3은 실습 1의 `ai_trend_md` 저장소를 쓴다.

## 0. 환경 준비 (연결·모델·문서)

In [2]:
import os, re
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
CONN = os.getenv("DATABASE_URL").replace("postgresql://", "postgresql+psycopg://")

from sqlalchemy import create_engine

engine = create_engine(CONN, pool_size=2, max_overflow=0)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # temperature=0: 재현성

DOCS = [
    "결제 오류 코드 E-4021은 카드 한도 초과를 뜻한다.",
    "결제 오류 코드 E-4022는 카드 정보 오류를 뜻한다.",
    "결제 오류 코드 E-4023은 잔액 부족을 뜻한다.",
    "로그인 오류 코드 L-1001은 비밀번호 불일치를 뜻한다.",
    "시스템 점검은 매주 일요일 새벽에 진행된다.",
    "환불은 구매 후 7일 이내 가능하다.",
]

# 한글·영문·숫자를 분리하는 토크나이저 (조사 분리)
def tok(t): return re.findall(r"[a-zA-Z]+|[0-9]+|[가-힣]+", t)

print("준비 완료")

준비 완료


## Part 1. 하이브리드 검색 — 기초

완성 코드를 직접 쳐서 키워드 검색(BM25)과 의미 검색(pgvector)을 각각 만들고 하나로 결합한다.

### 1-1. 키워드 리트리버(BM25)

한글은 조사가 붙어 정확 용어가 안 잡힌다 → 토크나이저로 분리한다.

In [3]:
# TODO: DOCS로 preprocess_func=tok인 BM25Retriever를 만들어 bm25에 저장하고 bm25.k = 3으로 설정하세요
bm25_retriever = BM25Retriever.from_texts(DOCS, preprocess_func=tok) # 토크나이저 설정 필수
bm25_retriever.k = 3


print("[BM25] E-4021 검색:")

result = bm25_retriever.invoke("E-4021 무슨 뜻입니까?")
result

[BM25] E-4021 검색:


[Document(metadata={}, page_content='결제 오류 코드 E-4021은 카드 한도 초과를 뜻한다.'),
 Document(metadata={}, page_content='환불은 구매 후 7일 이내 가능하다.'),
 Document(metadata={}, page_content='시스템 점검은 매주 일요일 새벽에 진행된다.')]

### 1-2. 의미 리트리버(pgvector)

Day03의 방식으로 의미 리트리버를 만든다.

In [7]:
# TODO: collection_name="hybrid_docs"인 PGVector store를 만들어 DOCS를 저장하고
#       search_kwargs={"k": 3}인 dense retriever로 변환하세요


store = PGVector(embeddings=embeddings, 
                 collection_name="hybrid_docs", 
                 connection=engine, 
                 use_jsonb=True)

store.add_texts(DOCS)

dense_retriever = store.as_retriever(search_kwargs={"k":3})

print("[의미 검색] E-4021 검색:")

for d in dense_retriever.invoke("E-4021 무슨 뜻이야?"):
    print(" -", d.page_content)




[의미 검색] E-4021 검색:
 - 결제 오류 코드 E-4022는 카드 정보 오류를 뜻한다.
 - 결제 오류 코드 E-4022는 카드 정보 오류를 뜻한다.
 - 결제 오류 코드 E-4022는 카드 정보 오류를 뜻한다.


### 1-3. 하이브리드로 결합

두 검색 결과를 RRF로 합친다.

In [27]:
# TODO: bm25·dense를 weights=[0.5, 0.5]로 결합하는 EnsembleRetriever를 hybrid에 만드세요
hybrid = EnsembleRetriever(retrievers=[bm25_retriever,dense_retriever], weights=[0.5, 0.5])

print("준비 완료")

준비 완료


### 1-4. 의미 단독 vs 하이브리드 비교

정확 용어(E-4021) 질의에서 의미 검색이 놓치는 걸 하이브리드가 잡는지 본다.

In [21]:
q = "E-4021 무슨 뜻이야?"
print("\n[하이브리드 top3]")

for d in hybrid.invoke(q):
    print(d.page_content)
    


[하이브리드 top3]
결제 오류 코드 E-4022는 카드 정보 오류를 뜻한다.
결제 오류 코드 E-4021은 카드 한도 초과를 뜻한다.
환불은 구매 후 7일 이내 가능하다.
시스템 점검은 매주 일요일 새벽에 진행된다.


### 1-5. 오류 대응 ① — 토크나이저 없이 BM25를 만들면?

`preprocess_func`를 안 주고 기본 방식(공백 기준)으로 나누면 어떻게 되는지 확인한다.

In [25]:
# TODO: preprocess_func 없이 DOCS로 BM25Retriever를 만들어 bm25_naive에 저장하고 bm25_naive.k = 3으로 설정하세요


print("[토크나이저 없는 BM25] E-4021 검색:")

bm25_naive = BM25Retriever.from_texts(DOCS) # 토크나이저 설정 필수
bm25_naive.k = 3

result = bm25_naive.invoke("E-4021 무슨 뜻입니까?")
result

    

[토크나이저 없는 BM25] E-4021 검색:


[Document(metadata={}, page_content='환불은 구매 후 7일 이내 가능하다.'),
 Document(metadata={}, page_content='시스템 점검은 매주 일요일 새벽에 진행된다.'),
 Document(metadata={}, page_content='로그인 오류 코드 L-1001은 비밀번호 불일치를 뜻한다.')]

> **주의:** 결과는 실제 실행에서 직접 확인한다. "E-4021은"처럼 조사가 코드 뒤에 바로 붙은 한글 문장은, 공백 기준 기본 토크나이저로는 "E-4021"과 "은"이 분리되지 않아 질문의 "E-4021"과 아예 매칭되지 않을 수 있다 — 예외 없이 조용히 순위에서 밀려나는(또는 아예 빠지는) 실패라 알아채기 더 어렵다.

### 1-6. 오류 대응 ② — 가중치 개수가 안 맞으면?

리트리버는 2개인데 `weights`를 1개만 주면 어떻게 되는지 확인한다.

In [ ]:
try:
    # TODO: retrievers=[bm25, dense]에 weights=[0.5](개수 불일치)로 EnsembleRetriever를 만들어보세요


    print("생성 성공(예상 밖)")
except Exception as e:
    print("오류 발생:", type(e).__name__, "-", e)

> **주의:** 결과는 실제 실행에서 직접 확인한다. `weights`의 개수는 반드시 `retrievers`의 개수와 같아야 한다 — 리트리버마다 얼마나 반영할지를 정하는 값이라, 개수가 안 맞으면 어느 리트리버에 어떤 가중치를 줄지 정할 수 없기 때문이다.

### 관찰 정리

- 하이브리드가 의미 검색 단독보다 나은 경우는 언제인가?
- 1-5·1-6에서 확인한 두 문제는 각각 '조용한 실패'와 '즉시 발생하는 오류' 중 어느 쪽에 가까운가?

## Part 2. 리랭킹 — 응용

넉넉히 모은 후보를 LLM 리랭커로 다시 순서 매겨 상위 품질을 높인다.

### 2-1. 후보 넉넉히 가져오기

놓침을 줄이려 하이브리드로 후보 6개를 모은다.

In [30]:
q = "E-4021 무슨 뜻이야?"
# TODO: hybrid.invoke(q)의 상위 6개를 cand에 저장하세요


print("[리랭킹 전 후보]")

    

[리랭킹 전 후보]


### 2-2. LLM 리랭커

각 후보의 관련도를 LLM이 0~10으로 매겨 재정렬한다.

In [29]:
def rerank(query, docs, top_n=3):
    scored = []
    for d in docs:
        r = llm.invoke(f"질문: {query}\n문서: {d.page_content}\n"
                       "이 문서가 질문에 얼마나 관련되나? 0~10 숫자만 답해라.").content
        scored.append((int(re.findall(r"\d+", r)[0]), d))
    scored.sort(key=lambda x: -x[0])
    return scored[:top_n]


### 2-3. 리랭킹 적용·비교

무관한 후보가 빠지고 정확한 조각이 위로 올라온다.

In [39]:
print("[리랭킹 후 top3 (점수)]")
candidate = hybrid.invoke(q)

for s, d in rerank(q,candidate):
    print(f" {s} - {d.page_content}")

[리랭킹 후 top3 (점수)]
 10 - 결제 오류 코드 E-4021은 카드 한도 초과를 뜻한다.
 1 - 결제 오류 코드 E-4022는 카드 정보 오류를 뜻한다.
 0 - 환불은 구매 후 7일 이내 가능하다.


### 관찰 정리

| 단계 | 상위 결과 | 특징 |
| --- | --- | --- |
| 리랭킹 전 | 후보 6개(무관 문장 섞임) | 넓게 모음(재현율) |
| 리랭킹 후 | 관련 높은 3개 | 정밀하게 추림(정밀도) |

### 2-4. 다른 도메인 반복 — 사내 규정 조항 번호 검색

같은 방법을 다른 도메인에도 적용해본다. 이번엔 조항 번호가 있는 사내 규정 문서다 — 오류 코드처럼 정확한 번호가 중요하다.

In [ ]:
DOCS2 = [
    "사내 규정 제12조는 연차 휴가 사용 기준을 정한다.",
    "사내 규정 제13조는 초과 근무 수당 지급 기준을 정한다.",
    "사내 규정 제14조는 출장비 정산 절차를 정한다.",
    "사내 규정 제21조는 재택근무 신청 절차를 정한다.",
    "사내 규정 제22조는 육아휴직 신청 자격을 정한다.",
    "회사 창립기념일은 매년 5월에 진행된다.",
]

# TODO: DOCS2로 bm25_2·store2·dense2·hybrid2를 만드세요
#       (bm25_2: preprocess_func=tok, k=3 / store2: collection_name="hybrid_docs2" / dense2: k=3 / hybrid2: weights=[0.5, 0.5])






q2 = "제13조가 뭐야?"


print("[리랭킹 전]")

    
print("\n[리랭킹 후 top3]")

    

### 관찰 정리

- 오류 코드 도메인과 조항 번호 도메인 모두에서, 하이브리드·리랭킹이 정확한 번호를 상위로 올렸는가?
- 리랭킹 비용(LLM 호출 수)은 후보 수와 어떤 관계인가?

### Part 3. 재순위·문맥 압축 — AI 트렌드 보고서

Part 2의 LLM 리랭커는 후보마다 LLM을 한 번씩 부른다. 이번에는 실제 보고서(실습 1의 `ai_trend_md`)로 두 가지를 더 본다.

| 기법 | 하는 일 |
| --- | --- |
| 로컬 리랭커 (Flashrank) | 내 PC에서 작은 모델로 순위를 다시 매긴다. API 비용이 없다 |
| 문맥 압축 (Contextual Compression) | 검색한 조각에서 질문과 관련된 부분만 남기거나, 관련 없는 조각을 뺀다 |

> **참고:** 실습 1(`Day13-1-NaiveRAG한계와문서전처리-실습.ipynb`)을 먼저 실행해 `ai_trend_md` 저장소를 만들어 둔다.

In [41]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor, LLMChainFilter, EmbeddingsFilter, DocumentCompressorPipeline)
from langchain_community.document_compressors import FlashrankRerank
from flashrank import Ranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

md_store = None


Q = "AI 에이전트가 기업 업무 방식에 어떤 변화를 가져올 것으로 전망되나요?"

ANSWER_MARK = "문서 처리"      # 답이 들어 있는 조각을 표시하려고 쓰는 문구 (보고서 17쪽)
def mark(d):
    return "★" if ANSWER_MARK in d.page_content else " "

candidates = None      # 1단계: 후보를 넓게

print("[벡터 검색 순서]")


[벡터 검색 순서]


★가 답(보고서 17쪽 "기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원…")이 들어 있는 조각이다. 재순위 뒤에 ★가 몇 번째로 가는지 본다.

#### 3-1. 로컬 리랭커 — Flashrank

Flashrank는 질문과 조각을 한 쌍씩 작은 모델(크로스 인코더)에 넣어 관련도 점수를 매긴다. 처음 실행할 때 모델(약 5MB)을 `flashrank_cache/` 폴더에 내려받는다.

In [ ]:

# TODO: flash로 candidates를 질문 Q 기준으로 재순위하세요 (compress_documents)




점수를 본다. 거의 모든 조각이 1.0에 가깝다. 이 모델(`ms-marco-TinyBERT`)은 **영어** 검색 데이터로 학습했기 때문에 한국어 조각의 차이를 구분하지 못한다. 그 결과 ★의 순위가 벡터 검색보다 나빠질 수 있다.

> **주의:** 다국어 모델(`ms-marco-MultiBERT-L-12`, 약 100MB)도 이 보고서에서 점수가 0.999~1.0에 몰려 구분이 약했다(2026-09-25 확인). 로컬 리랭커는 **문서의 언어와 분야에 맞는 모델**이어야 효과가 있다. 쓰기 전에 이렇게 점수 분포부터 확인한다.

#### 3-2. 같은 후보를 LLM 리랭커로

Part 2의 `rerank` 함수를 같은 10개 후보에 쓴다.

| 구분 | Flashrank (TinyBERT) | LLM 리랭커 |
| --- | --- | --- |
| 비용 | 없음 (내 PC에서 실행) | 후보 수만큼 LLM 호출 |
| 속도 | 빠르다 | 후보 수에 비례해 느리다 |
| 한국어 | 점수가 1.0 근처에 몰린다 | 0~10점으로 차이가 난다 |

출력에서 ★의 순위와 걸린 시간을 비교한다.

#### 3-3. 문맥 압축 ① — 관련 부분만 뽑기 (LLMChainExtractor)

검색된 조각은 800자 안팎이다. 그중 질문과 관련된 문장은 일부다. `LLMChainExtractor`는 조각마다 LLM에게 "질문과 관련된 부분만 그대로 뽑아라"라고 시킨다. `ContextualCompressionRetriever`는 리트리버 뒤에 이런 압축기를 붙이는 틀이다.

In [ ]:
# TODO: base_retriever 뒤에 LLMChainExtractor를 붙인 압축 검색기를 만드세요 (ContextualCompressionRetriever)


#### 3-4. 문맥 압축 ② — 관련 없는 조각 빼기

| 압축기 | 판단 방법 | 비용 |
| --- | --- | --- |
| `LLMChainFilter` | LLM이 조각마다 "관련 있다/없다"를 답한다 | 조각 수만큼 LLM 호출 |
| `EmbeddingsFilter` | 질문과 조각의 임베딩 유사도가 기준값(`similarity_threshold`) 이상인 것만 남긴다 | 임베딩만 (저렴) |

`similarity_threshold`는 코사인 **유사도**(1에 가까울수록 비슷함) 기준이다. 실습 1의 `similarity_search_with_score`가 보여 준 **거리**(0에 가까울수록 비슷함)와 방향이 반대다.

출력에서 기준값에 따라 남는 조각 수를 본다. 0.5에서는 답이 있는 17쪽 조각만 남고, 0.6에서는 하나도 남지 않는다. 기준값을 조금만 높여도 근거가 모두 사라질 수 있으므로, 문서마다 출력을 보고 정한다. `LLMChainFilter`는 네 조각을 모두 "관련 있다"로 판단했다.

#### 3-5. 압축기 이어 붙이기 → RAG 체인

`DocumentCompressorPipeline`으로 압축기를 차례로 잇는다. 싼 필터(`EmbeddingsFilter`)로 관련 없는 조각을 먼저 빼고, 남은 조각에만 비싼 추출기(`LLMChainExtractor`)를 쓴다.

In [ ]:
# TODO: EmbeddingsFilter(기준 0.5) → LLMChainExtractor 순서로 잇는 파이프라인을 만드세요


#### 관찰

| 확인 | 결과 |
| --- | --- |
| 3-1 Flashrank | 10개 모두 0.9997~0.9999점으로 몰린다. 벡터 검색 1위였던 ★가 4위로 밀렸다 |
| 3-2 LLM 리랭커 | ★가 9점으로 1위, 나머지는 3~8점으로 차이가 난다. 대신 LLM을 10번 불러 수 초가 걸린다 |
| 3-3 추출 | 근거가 약 2,700자에서 1,000자 안팎으로 줄었다(실행마다 조금씩 다르다). 줄어든 만큼 답변 모델에 넣는 토큰이 준다 |
| 3-4 필터 | LLM 필터는 네 조각을 모두 남겼고, 임베딩 필터는 기준값 0.5에서 1개, 0.6에서 0개를 남겼다 |
| 3-5 파이프라인 | 근거가 약 2,700자에서 약 200자로 줄었는데도 같은 핵심 내용으로 답했다 |

> **참고:** 어떤 조합이 이 문서에서 실제로 나은지는 실습 4의 평가 세트로 잰다.

## Part 3. 미니 프로젝트 — RAG 평가

검색 정확도로 기본 RAG와 개선 RAG를 점수화해 비교한다.

### 3-1. 평가기 정의

검색이 정답 문서를 1위(top-1)로 가져왔는지를 정답 문구로 확인한다.

In [ ]:
# (질문, 정답 문구) 세트
eval_set = [
    ("E-4021 무슨 뜻이야?", "카드 한도 초과"),
    ("L-1001은 무슨 오류야?", "비밀번호 불일치"),
    ("환불은 언제까지 되나?", "구매 후 7일 이내"),
]

# TODO: retriever로 question을 검색해(rerank_on이면 상위 6개를 rerank까지 적용) top-1 문서의
#       page_content를 반환하는 top1 함수를 만드세요
#       (def top1(retriever, question, rerank_on=False): ...)




### 3-2. 기본 vs 개선 점수 비교

기본은 의미 검색만, 개선은 하이브리드+리랭킹. 같은 질문 세트로 검색 정확도를 낸다.

In [ ]:
# TODO: name·retriever·rerank_on을 받아 eval_set의 각 (question, expected)에 대해
#       top1으로 ctx를 구하고, expected가 ctx에 있으면 1 아니면 0으로 hit을 누적해
#       평균 정확도를 출력하는 evaluate 함수를 만드세요
#       (def evaluate(name, retriever, rerank_on): ...)



### 관찰

| RAG | 검색 정확도 | E-4021 top-1 |
| --- | --- | --- |
| 기본(의미만) | ? | ? |
| 개선(하이브리드+리랭킹) | ? | ? |

> **참고:** 실제 점수는 실행 결과에서 직접 채운다. 검색이 엉뚱한 조각을 1위로 두면 답도 틀린다 — 검색·순위 품질이 RAG 품질을 좌우한다.

## Part 3-확장. M3 전체 통합 — 면접 코치 지식 검색을 정량 평가

Day01(M3)~Day03(M3)에서 쌓아 온 면접 코치의 채용 공고 지식 검색을, 오늘 배운 하이브리드+리랭킹과 평가기로 **직접 점수를 매겨** 정말 개선되는지 확인한다.

### 채용 공고 하이브리드 리트리버 (Day01~03 재사용)

In [ ]:
job_postings = [
    "백엔드 개발자: Python/Java 기반 서버 개발, RESTful API 설계, DB 최적화 경험 우대",
    "프론트엔드 개발자: React/TypeScript, 반응형 UI 구현, 웹 접근성 이해 우대",
    "데이터 분석가: SQL/Python 데이터 분석, 대시보드 제작, 통계적 사고 우대",
    "마케터: 콘텐츠 기획, 데이터 기반 캠페인 분석, SNS 채널 운영 경험 우대",
]

# TODO: job_postings로 job_bm25(k=1)·job_store(collection_name="job_postings_eval")·
#       job_dense(k=1)·job_hybrid(weights=[0.5, 0.5])를 만드세요





print("채용 공고 검색기 준비 완료")

### 채용 공고 검색 평가 세트

In [ ]:
job_eval_set = [
    ("데이터 분석 직무를 준비하고 있어요.", "데이터 분석가"),
    ("React를 사용하는 프론트엔드 개발자가 되고 싶어요.", "프론트엔드 개발자"),
    ("서버 API를 개발하는 백엔드 개발자 직무에 관심 있어요.", "백엔드 개발자"),
    ("SNS 마케팅 캠페인을 기획하는 직무를 찾고 있어요.", "마케터"),
]

# TODO: job_eval_set의 각 (question, expected)에 대해 retriever로 검색한(rerank_on이면 리랭킹까지 적용)
#       top-1 문서에 expected가 있는지로 hit을 누적해 평균 정확도를 출력하는 evaluate_jobs 함수를 만드세요





### M3(지식·기억) 모듈 최종 정리

**확인 질문**
- 면접 코치의 채용 공고 검색에서도 오류 코드 예시와 같은 개선이 나타났는가? 나타나지 않았다면 왜일까?
- Day01(임베딩)부터 오늘(하이브리드·리랭킹·평가)까지, 지식 검색은 어떤 순서로 발전해 왔는가?
- 로컬 리랭커와 LLM 리랭커, 문맥 압축은 각각 어떤 비용을 내고 무엇을 얻는가?

## 확인 문제

1. 하이브리드 검색이 의미 검색 단독보다 나은 경우는 언제인가?
2. 1-5·1-6에서 확인한 것처럼, '조용한 실패'와 '즉시 발생하는 오류'는 각각 어떻게 다른가?
3. Part 2와 Part 2의 도메인 반복에서, 정확한 번호를 찾는 원리가 도메인이 달라도 그대로 통했는가?
4. 검색 정확도(Context Precision)를 재는 방법은 무엇인가?
5. Part 4-확장에서 면접 코치의 검색 성능을 실제로 측정해 보니 어떤 결론을 얻었는가?
6. 3-1에서 Flashrank 점수가 모두 1.0 근처에 몰린 이유는? 로컬 리랭커를 고를 때 무엇을 확인해야 하나?
7. `LLMChainExtractor`와 `EmbeddingsFilter`는 각각 무엇을 기준으로 근거를 줄이며, 비용은 어떻게 다른가?
8. 3-5에서 필터를 추출기보다 앞에 둔 이유는?